# IEEE-CIS Fraud Detection: Memory-Aware Baseline

This experiment joins transaction and identity data, applies a stratified split, encodes categorical features consistently, and reports validation ROC-AUC. Paths and model settings are read from `config/config.yaml`.

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'config.yaml').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
with (PROJECT_ROOT / 'config' / 'config.yaml').open() as config_file:
    CONFIG = yaml.safe_load(config_file)
SEED = CONFIG['project']['random_seed']
RAW_DIR = PROJECT_ROOT / CONFIG['data']['raw_dir']
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data directory: {RAW_DIR}')

Project root: /Users/jaideepsharma/Documents/Projects/FRAUD_IEEE_CIS
Raw data directory: /Users/jaideepsharma/Documents/Projects/FRAUD_IEEE_CIS/input_data/ieee-fraud-detection


In [2]:
def reduce_memory_usage(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns while retaining missing values."""
    for column in dataframe.columns:
        if pd.api.types.is_float_dtype(dataframe[column].dtype):
            dataframe[column] = pd.to_numeric(dataframe[column], downcast='float')
        elif pd.api.types.is_integer_dtype(dataframe[column].dtype):
            dataframe[column] = pd.to_numeric(dataframe[column], downcast='integer')
    return dataframe

def load_training_data() -> pd.DataFrame:
    """Load and validate the transaction/identity one-to-one join."""
    transactions = pd.read_csv(RAW_DIR / CONFIG['data']['train_transaction'])
    identity = pd.read_csv(RAW_DIR / CONFIG['data']['train_identity'])
    merged = transactions.merge(identity, on=CONFIG['data']['id_column'], how='left', validate='one_to_one')
    return reduce_memory_usage(merged)

data = load_training_data()
target_column = CONFIG['data']['target_column']
print(f'Loaded shape: {data.shape}; fraud rate: {data[target_column].mean():.4%}')

Loaded shape: (590540, 434); fraud rate: 3.4990%


## Stratified split and categorical preparation

Categorical levels are learned from train and validation values, then represented as integer codes. Missing values receive a numeric sentinel.

In [3]:
y = data.pop(target_column).astype('int8')
data = data.drop(columns=[CONFIG['data']['id_column']], errors='ignore')
X_train, X_valid, y_train, y_valid = train_test_split(data, y, test_size=CONFIG['validation']['test_size'], stratify=y, random_state=SEED)

def encode_categoricals(train: pd.DataFrame, valid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply identical category codes to train and validation partitions."""
    train_encoded, valid_encoded = train.copy(), valid.copy()
    for column in train_encoded.columns:
        if train_encoded[column].dtype == 'object':
            combined = pd.concat([train_encoded[column], valid_encoded[column]], ignore_index=True).astype('category')
            train_encoded[column] = combined.iloc[:len(train_encoded)].cat.codes.astype('int32').to_numpy()
            valid_encoded[column] = combined.iloc[len(train_encoded):].cat.codes.astype('int32').to_numpy()
    return train_encoded.fillna(-999), valid_encoded.fillna(-999)

X_train, X_valid = encode_categoricals(X_train, X_valid)
print(f'Train: {X_train.shape}; validation: {X_valid.shape}')

Train: (472432, 432); validation: (118108, 432)


In [4]:
model_config = CONFIG['model']
scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
try:
    from lightgbm import LGBMClassifier
    model = LGBMClassifier(n_estimators=model_config['n_estimators'], learning_rate=model_config['learning_rate'], num_leaves=model_config['num_leaves'], max_depth=model_config['max_depth'], subsample=model_config['subsample'], colsample_bytree=model_config['colsample_bytree'], objective='binary', scale_pos_weight=scale_pos_weight, random_state=SEED, n_jobs=model_config['n_jobs'], verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[])
except ImportError as error:
    raise ImportError('Install requirements.txt to run the LightGBM baseline.') from error
validation_probabilities = model.predict_proba(X_valid)[:, 1]
baseline_roc_auc = roc_auc_score(y_valid, validation_probabilities)
print(f'Baseline validation ROC-AUC: {baseline_roc_auc:.6f}')

Baseline validation ROC-AUC: 0.938988


In [10]:
# importance = pd.DataFrame({
#     "feature": X_train.columns,
#     "split_count": model.feature_importances_,
#     "gain": model.booster_.feature_importance(importance_type="gain"),
# })

# importance.sort_values("gain", ascending=False).head(30)

In [11]:
# contributions = model.predict(X_valid, pred_contrib=True)

# contribution_columns = list(X_valid.columns) + ["bias"]
# contributions = pd.DataFrame(
#     contributions,
#     columns=contribution_columns,
#     index=X_valid.index,
# )

# contributions.abs().mean().sort_values(ascending=False).head(30)

## Interpretation

The printed ROC-AUC is a quick benchmark only. Before trusting improvements, compare stronger validation schemes, inspect temporal and identity distribution shift, and save the model plus preprocessing metadata together.

## Stratified K-fold validation

A single train/validation split gives one estimate of generalization. Stratified K-fold validation repeats the experiment across several class-balanced partitions. Preprocessing is fitted separately inside each fold, and the out-of-fold probabilities provide an aggregate ROC-AUC using every row exactly once for validation.

In [8]:
from sklearn.model_selection import StratifiedKFold
import numpy as np


def encode_fold(train: pd.DataFrame, valid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit category levels on the fold's training data only."""
    train_encoded, valid_encoded = train.copy(), valid.copy()
    for column in train_encoded.columns:
        if train_encoded[column].dtype == "object" or isinstance(train_encoded[column].dtype, pd.CategoricalDtype):
            categories = pd.Index(train_encoded[column].dropna().unique())
            category_dtype = pd.api.types.CategoricalDtype(categories=categories)
            train_encoded[column] = train_encoded[column].astype(category_dtype).cat.codes.astype("int32")
            valid_encoded[column] = valid_encoded[column].astype(category_dtype).cat.codes.astype("int32")
    return train_encoded.fillna(-999), valid_encoded.fillna(-999)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probabilities = np.zeros(len(data), dtype="float32")
fold_scores = []

for fold_number, (train_indices, valid_indices) in enumerate(cv.split(data, y), start=1):
    fold_train, fold_valid = encode_fold(
        data.iloc[train_indices],
        data.iloc[valid_indices],
    )
    fold_y_train, fold_y_valid = y.iloc[train_indices], y.iloc[valid_indices]
    fold_scale_pos_weight = float((fold_y_train == 0).sum() / max((fold_y_train == 1).sum(), 1))
    fold_model = LGBMClassifier(
        n_estimators=model_config["n_estimators"],
        learning_rate=model_config["learning_rate"],
        num_leaves=model_config["num_leaves"],
        max_depth=model_config["max_depth"],
        subsample=model_config["subsample"],
        colsample_bytree=model_config["colsample_bytree"],
        objective="binary",
        scale_pos_weight=fold_scale_pos_weight,
        random_state=SEED + fold_number,
        n_jobs=model_config["n_jobs"],
        verbosity=-1,
    )
    fold_model.fit(fold_train, fold_y_train)
    fold_probabilities = fold_model.predict_proba(fold_valid)[:, 1]
    oof_probabilities[valid_indices] = fold_probabilities
    fold_auc = roc_auc_score(fold_y_valid, fold_probabilities)
    fold_scores.append(fold_auc)
    print(f"Fold {fold_number}: ROC-AUC = {fold_auc:.6f}")

cv_results = pd.DataFrame({"fold": range(1, len(fold_scores) + 1), "roc_auc": fold_scores})
print(f"Mean ROC-AUC: {np.mean(fold_scores):.6f}")
print(f"Std ROC-AUC:  {np.std(fold_scores, ddof=1):.6f}")
print(f"OOF ROC-AUC:  {roc_auc_score(y, oof_probabilities):.6f}")
cv_results

Fold 1: ROC-AUC = 0.940529
Fold 2: ROC-AUC = 0.940296
Fold 3: ROC-AUC = 0.937794
Fold 4: ROC-AUC = 0.938395
Fold 5: ROC-AUC = 0.938829
Mean ROC-AUC: 0.939169
Std ROC-AUC:  0.001196
OOF ROC-AUC:  0.939121


,fold,roc_auc
0,1,0.940529
1,2,0.940296
2,3,0.937794
3,4,0.938395
4,5,0.938829


## Hyperparameter tuning

The original validation set remains untouched during tuning. Randomized search uses three stratified folds within `X_train` and optimizes ROC-AUC across tree complexity, learning rate, regularization, and sampling parameters. The selected configuration is then refit on all training rows and evaluated once on `X_valid`.

In [9]:
from sklearn.model_selection import RandomizedSearchCV


tuning_model = LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

parameter_distributions = {
    "n_estimators": [200, 300, 500],
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 8, 12, 16],
    "min_child_samples": [20, 50, 100, 200],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "subsample_freq": [1],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0, 10.0],
}

tuning_search = RandomizedSearchCV(
    estimator=tuning_model,
    param_distributions=parameter_distributions,
    n_iter=6,
    scoring="roc_auc",
    cv=3,
    refit=True,
    random_state=SEED,
    n_jobs=1,
    return_train_score=False,
    verbose=1,
)
tuning_search.fit(X_train, y_train)

tuning_results = (
    pd.DataFrame(tuning_search.cv_results_)
    .sort_values("rank_test_score")
    [["rank_test_score", "mean_test_score", "std_test_score", "params"]]
)

tuned_validation_probabilities = tuning_search.best_estimator_.predict_proba(X_valid)[:, 1]
tuned_validation_auc = roc_auc_score(y_valid, tuned_validation_probabilities)
print(f"Best tuning CV ROC-AUC: {tuning_search.best_score_:.6f}")
print(f"Tuned holdout ROC-AUC:   {tuned_validation_auc:.6f}")
print(f"Baseline holdout ROC-AUC: {baseline_roc_auc:.6f}")
print(f"Holdout change:           {tuned_validation_auc - baseline_roc_auc:+.6f}")
print("Best parameters:")
print(tuning_search.best_params_)
tuning_results

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best tuning CV ROC-AUC: 0.961221
Tuned holdout ROC-AUC:   0.971139
Baseline holdout ROC-AUC: 0.938988
Holdout change:           +0.032151
Best parameters:
{'subsample_freq': 1, 'subsample': 0.7, 'reg_lambda': 10.0, 'num_leaves': 127, 'n_estimators': 500, 'min_child_samples': 20, 'max_depth': 12, 'learning_rate': 0.08, 'colsample_bytree': 1.0}


,rank_test_score,mean_test_score,std_test_score,params
3,1,0.961221,0.000086,"{'subsample_freq': 1, 'subsample': 0.7, 'reg_l..."
5,2,0.958876,0.000378,"{'subsample_freq': 1, 'subsample': 0.8, 'reg_l..."
0,3,0.952802,0.000769,"{'subsample_freq': 1, 'subsample': 1.0, 'reg_l..."
4,4,0.951732,0.000567,"{'subsample_freq': 1, 'subsample': 0.7, 'reg_l..."
1,5,0.926130,0.001724,"{'subsample_freq': 1, 'subsample': 0.7, 'reg_l..."
2,6,0.921491,0.001565,"{'subsample_freq': 1, 'subsample': 0.9, 'reg_l..."
